In [1]:
import json
from tqdm import tqdm

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

import torch

In [2]:
if torch.backends.mps.is_available():
    print('SI')
else:
    print('NO')

SI


In [3]:
with open("database_chunk_riferimenti.json", 'r', encoding='utf-8') as d:
    database = json.load(d)


documenti = []

for sentenza in database:
    for ind, chunk in enumerate(sentenza['chunks'].values()):
        nodi = [r.get('nodo') for r in chunk.get('riferimenti', []) if r.get('nodo')]
        doc = Document(page_content=chunk['chunk'],
                       metadata={'id':f"{sentenza['ID']}/{ind}",
                                 'sentenza_id': sentenza['ID'],
                                 'materia': sentenza['metadati'].get('materia'),
                                 'tipo_pronuncia': sentenza['metadati'].get('tipo_pronuncia'),
                                 'sezione': sentenza['metadati'].get('sezione'),
                                 'presidente': sentenza['metadati'].get('presidente'),
                                 'relatore': sentenza['metadati'].get('relatore'),
                                 'data_pubblicazione': sentenza['metadati'].get('data_pubblicazione'),
                                 'rif': nodi if nodi else None}
                                 )
        documenti.append(doc)

In [ ]:
embedding_model_name = ("dlicari/Italian-Legal-BERT")
                   
embedding_model = HuggingFaceEmbeddings(model_name=embedding_model_name, model_kwargs={"device": "mps"})

vector_store = Chroma(collection_name='CassCiv',
                      embedding_function=embedding_model,
                      persist_directory='/Users/maxiorsini/Cassazione_Hybrid_RAG/cass_vector_DB')

batch_size = 10

for doc in tqdm(documenti):
    list_doc = [doc]
    vector_store.add_documents(list_doc)
